# Asteroid Orbital Visual — Automated Pipeline
Run **Cell → Run All** to execute the full pipeline end-to-end:
1. Fetch raw NASA CNEOS data from GitHub (falls back to local if present)
2. Flatten into `asteroids_flat.csv`
3. Verify Node.js / npm
4. Install `powerbi-visuals-tools` globally (skipped if already installed)
5. Install visual npm dependencies
6. Install the pbiviz developer certificate
7. Start the pbiviz dev server (background)
8. Launch Power BI Desktop
9. Print field-mapping checklist + Power BI Web connector URL
10. Preview and profile the flattened data

In [ ]:
import subprocess, sys, os, time, threading, glob
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
VISUAL_DIR   = NOTEBOOK_DIR / 'asteroid-orbital-visual'
DATA_SCRIPT  = NOTEBOOK_DIR / 'flatten_asteroids.py'
RAW_CSV      = NOTEBOOK_DIR / 'asteroids_data.csv'
FLAT_CSV     = NOTEBOOK_DIR / 'asteroids_flat.csv'

GITHUB_REPO      = 'jdstigma/nasa-asteroids'
GITHUB_BRANCH    = 'main'
GITHUB_RAW_BASE  = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}'
RAW_CSV_URL      = f'{GITHUB_RAW_BASE}/asteroids_data.csv'
FLAT_CSV_URL     = f'{GITHUB_RAW_BASE}/asteroids_flat.csv'

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout.strip() or r.stderr.strip(), r.returncode

def ok(msg):   print(f'  ✅ {msg}')
def err(msg):  print(f'  ❌ {msg}')
def info(msg): print(f'  ℹ  {msg}')

print('Config')
print(f'  Notebook     : {NOTEBOOK_DIR}')
print(f'  GitHub repo  : {GITHUB_REPO}')
print(f'  Raw CSV URL  : {RAW_CSV_URL}')
print(f'  Flat CSV URL : {FLAT_CSV_URL}')

## Step 1 — Fetch raw data from GitHub

In [ ]:
import requests

print('Step 1 — Fetch raw NASA data from GitHub')

def download_csv(url, dest_path, label):
    """Stream a CSV from a URL to disk with a progress indicator."""
    info(f'Downloading {label} from GitHub...')
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    downloaded = 0
    with open(dest_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 256):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f'    {pct:5.1f}%  ({downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB)', end='\r')
    print()
    ok(f'{label} saved → {dest_path.name}  ({dest_path.stat().st_size/1024/1024:.1f} MB)')

if RAW_CSV.exists():
    ok(f'Local raw file found ({RAW_CSV.stat().st_size/1024/1024:.1f} MB) — skipping download')
    info('Delete asteroids_data.csv to force a fresh download from GitHub.')
else:
    download_csv(RAW_CSV_URL, RAW_CSV, 'asteroids_data.csv')

## Step 2 — Flatten into asteroids_flat.csv

In [ ]:
print('Step 2 — Flatten NASA CNEOS data')
result = subprocess.run([sys.executable, str(DATA_SCRIPT)], capture_output=True, text=True)
if result.returncode != 0:
    err('Data prep failed'); print(result.stderr); raise RuntimeError
print(result.stdout.strip())
rows = sum(1 for _ in open(FLAT_CSV)) - 1
ok(f'asteroids_flat.csv — {rows:,} close-approach rows  ({FLAT_CSV.stat().st_size/1024/1024:.1f} MB)')

## Step 3 — Verify Node.js & npm

In [ ]:
print('Step 3 — Node.js / npm')
node_ver, node_rc = run('node --version')
npm_ver,  npm_rc  = run('npm --version')
if node_rc != 0 or npm_rc != 0:
    err('Node.js not found. Install from https://nodejs.org then restart the kernel.')
    raise EnvironmentError
ok(f'Node {node_ver}  |  npm {npm_ver}')

## Step 4 — Install powerbi-visuals-tools globally

In [ ]:
print('Step 4 — pbiviz CLI')
ver, rc = run('pbiviz --version')
if rc == 0:
    ok(f'pbiviz {ver} already installed')
else:
    info('Installing powerbi-visuals-tools globally...')
    out, rc = run('npm install -g powerbi-visuals-tools')
    if rc != 0: err(out); raise RuntimeError
    ver, _ = run('pbiviz --version')
    ok(f'pbiviz {ver} installed')

## Step 5 — Install visual npm dependencies

In [ ]:
print('Step 5 — npm install')
node_modules = VISUAL_DIR / 'node_modules'
if node_modules.exists():
    ok('node_modules already present — skipping')
else:
    info('Running npm install...')
    out, rc = run('npm install', cwd=str(VISUAL_DIR))
    if rc != 0: err(out); raise RuntimeError
    ok('Dependencies installed')

## Step 6 — Developer certificate

In [ ]:
print('Step 6 — Developer certificate')
cert_path = Path.home() / 'pbiviz-certs' / 'PowerBICustomVisualTest_public.pfx'
if cert_path.exists():
    ok(f'Certificate already exists')
else:
    info('Generating certificate...')
    out, rc = run('pbiviz install-cert', cwd=str(VISUAL_DIR))
    print(out)
    if cert_path.exists():
        ok('Certificate generated')
        passphrase = [l for l in out.splitlines() if 'Passphrase' in l]
        pw = passphrase[0].split()[-1] if passphrase else 'SEE ABOVE'
        print()
        print('  ⚠  ACTION REQUIRED — run this in an Admin PowerShell to trust the cert:')
        print(f"  Import-PfxCertificate -FilePath '{cert_path}' -CertStoreLocation Cert:\\CurrentUser\\Root -Password (ConvertTo-SecureString '{pw}' -Force -AsPlainText)")
    else:
        err('Certificate generation failed — check output above')

## Step 7 — Start the pbiviz dev server

In [ ]:
print('Step 7 — pbiviz dev server')

server_proc = subprocess.Popen(
    'pbiviz start', shell=True, cwd=str(VISUAL_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

server_lines = []
def _stream():
    for line in server_proc.stdout:
        server_lines.append(line.rstrip())
threading.Thread(target=_stream, daemon=True).start()
time.sleep(8)

errors    = [l for l in server_lines if 'error' in l.lower() and 'deprecation' not in l.lower()]
listening = any('8080' in l or 'listening' in l.lower() for l in server_lines)
compiled  = any('compiled successfully' in l for l in server_lines)

if errors and not compiled:
    for l in server_lines[-20:]: print(f'    {l}')
    err('Errors detected')
elif listening and compiled:
    ok('Dev server running on https://localhost:8080  |  webpack compiled successfully')
else:
    info('Server still starting...')
    for l in server_lines[-10:]: print(f'    {l}')

def kill_server():
    server_proc.terminate()
    print('Dev server stopped.')

## Step 8 — Launch Power BI Desktop

In [ ]:
print('Step 8 — Launch Power BI Desktop')

pbi_candidates = [
    r'C:\Program Files\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    r'C:\Program Files (x86)\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    str(Path.home() / 'AppData/Local/Microsoft/WindowsApps/PBIDesktop.exe'),
]
pbi_candidates += glob.glob(
    str(Path.home() / 'AppData/Local/Microsoft/WindowsApps/**/PBIDesktop.exe'), recursive=True
)
pbi_exe = next((p for p in pbi_candidates if Path(p).exists()), None)
pbix_files = list(NOTEBOOK_DIR.glob('*.pbix'))
pbix_arg   = str(pbix_files[0]) if pbix_files else None

if pbi_exe:
    cmd = [pbi_exe] + ([pbix_arg] if pbix_arg else [])
    subprocess.Popen(cmd)
    label = Path(pbix_arg).name if pbix_arg else 'blank project'
    ok(f'Launched Power BI Desktop — {label}')
else:
    err('Power BI Desktop not found.')
    info('Install from https://powerbi.microsoft.com/desktop or the Microsoft Store, then re-run this cell.')

## Step 9 — Field mapping checklist & Web connector URL

### Option A — Local file (fastest)
Home → Get data → Text/CSV → pick `asteroids_flat.csv`

### Option B — Live from GitHub (always up to date)
Home → Get data → Web → paste the URL printed below

In [ ]:
import pandas as pd

print('Step 9 — Power BI connection options')
print()
print('  Power BI Web connector URL (paste into Home → Get data → Web):')
print(f'  {FLAT_CSV_URL}')
print()
print('  Field mapping — drag these into the Developer Visual "Data Fields" bucket:')
print('  ' + '-' * 58)

FIELD_MAP = [
    ('name',                  'Asteroid identity'),
    ('short_name',            'Label in tooltips'),
    ('potentially_hazardous', 'Red color + glow pulse'),
    ('diameter_max_m',        'Dot size'),
    ('semi_major_axis',       'Orbit radius (AU)'),
    ('eccentricity',          'Orbit shape'),
    ('perihelion_argument',   'Orbit orientation'),
    ('miss_distance_au',      'Closest approach distance (tooltip)'),
    ('close_approach_date',   'Approach date (tooltip)'),
    ('velocity_km_s',         'Approach speed (tooltip)'),
    ('orbiting_body',         'Which planet approached'),
    ('magnitude',             'Brightness (tooltip)'),
    ('orbit_class_type',      'Class label (tooltip)'),
]
for col, role in FIELD_MAP:
    print(f'  {col:<30} →  {role}')
print()
print('  Formatting pane options:')
print('    Show All 8 Planets    — toggle outer planets on/off')
print('    Animation Speed       — days per frame (default 5)')
print('    Hazardous Only        — show only red asteroids')

## Step 10 — Data profile

In [ ]:
import matplotlib.pyplot as plt

print('Step 10 — Loading from GitHub for profile...')
df = pd.read_csv(FLAT_CSV_URL, low_memory=False)
print(f'  Loaded {len(df):,} rows from GitHub')

df['miss_distance_au']    = pd.to_numeric(df['miss_distance_au'],    errors='coerce')
df['velocity_km_s']       = pd.to_numeric(df['velocity_km_s'],       errors='coerce')
df['diameter_max_m']      = pd.to_numeric(df['diameter_max_m'],      errors='coerce')
df['close_approach_date'] = pd.to_datetime(df['close_approach_date'], errors='coerce')

print()
print(f'  Asteroids     : {df["name"].nunique():,} unique')
print(f'  Hazardous     : {(df["potentially_hazardous"]=="True").sum():,} approaches')
print(f'  Date range    : {df["close_approach_date"].min().date()} → {df["close_approach_date"].max().date()}')
print(f'  Orbiting body : {df["orbiting_body"].value_counts().to_dict()}')
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('NASA CNEOS Asteroid Close Approaches', fontsize=13, fontweight='bold')

close = df[df['miss_distance_au'] < 0.5]
axes[0].hist(close['miss_distance_au'].dropna(), bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Miss Distance (< 0.5 AU)')
axes[0].set_xlabel('Miss Distance (AU)')
axes[0].set_ylabel('Approach count')

df['decade'] = (df['close_approach_date'].dt.year // 10 * 10)
decade_counts = df['decade'].value_counts().sort_index()
axes[1].bar(decade_counts.index.astype(str), decade_counts.values, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].set_title('Close Approaches by Decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

haz     = df[df['potentially_hazardous'] == 'True']['velocity_km_s'].dropna()
non_haz = df[df['potentially_hazardous'] == 'False']['velocity_km_s'].dropna()
axes[2].hist(non_haz, bins=40, alpha=0.6, color='gray',    label='Non-hazardous', edgecolor='white', linewidth=0.2)
axes[2].hist(haz,     bins=40, alpha=0.8, color='crimson',  label='Hazardous',    edgecolor='white', linewidth=0.2)
axes[2].set_title('Velocity by Hazard Status')
axes[2].set_xlabel('Velocity (km/s)')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'assets' / 'data_profile.png', dpi=120, bbox_inches='tight')
plt.show()
print('  Profile chart saved to assets/data_profile.png')

In [ ]:
# Top 10 closest Earth approaches
earth = df[df['orbiting_body'] == 'Earth'].copy()
closest = (earth.sort_values('miss_distance_au')
               [['name', 'close_approach_date', 'miss_distance_au', 'velocity_km_s', 'potentially_hazardous']]
               .drop_duplicates('name')
               .head(10)
               .reset_index(drop=True))
closest.index += 1
print('Top 10 closest Earth approaches (one per asteroid):')
closest

In [ ]:
# Stop the dev server when done with Power BI
# kill_server()